# Train TIGER on Colab

Before running:
1. Top right: **Select Kernel → Colab → New Colab Server → GPU** (the free tier is usually a T4).
2. In the VS Code Explorer, right-click `tiger.py`, `split.npz` and `semantic_ids.npy` → **Upload to Colab** (they go to `/content` on the server).
3. Click the Colab button in the notebook toolbar → **Mount Google Drive to Server**, run the cell it inserts and authorize. With Drive mounted, checkpoints and logs are saved to Drive, so training can resume after a disconnect.

Then run the cells below from top to bottom.

In [ ]:
# Check that a GPU is attached
!nvidia-smi

Tue Nov 17 00:44:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Colab ships with torch; only install the same transformers version as locally
!pip install -q transformers==5.23.0

In [7]:
import os
import shutil

# With Drive mounted, train inside Drive (checkpoints survive a disconnect); otherwise use /content (files are lost on disconnect)
WORK = "/content/drive/MyDrive/DSC200_tiger" if os.path.exists("/content/drive/MyDrive") else "/content"
os.makedirs(WORK, exist_ok=True)

# Copy the files uploaded to /content into the working directory (files already in Drive are not copied again)
for f in ["tiger.py", "split.npz", "semantic_ids.npy"]:
    src = f"/content/{f}"
    if WORK != "/content" and os.path.exists(src):
        shutil.copy(src, WORK)
    assert os.path.exists(f"{WORK}/{f}"), f"{f} not found: right-click it and upload it to Colab first"

os.chdir(WORK)
print("Working directory:", WORK)
print(sorted(os.listdir(WORK)))

Working directory: /content/drive/MyDrive/DSC200_tiger
['semantic_ids.npy', 'split.npz', 'tiger.py']


In [ ]:
# Start training. Each epoch prints one line with the loss and val metrics and appends it to logs/tiger.log
# After a disconnect: reconnect, rerun the cells above and this one; training resumes from the latest checkpoint
!python tiger.py

Device: cuda | training precision: torch.float16 | vocab size: 2824 | level-4 codes: 54
Training samples: 756798 | val users: 73594 | test users: 74087 | input length: 81
Parameters: 4.82M
Epoch  1 | loss 4.5338 | Val HR@10 0.3960 | Val NDCG@10 0.2458 | train 362s, eval 409s
Epoch  2 | loss 3.8251 | Val HR@10 0.4539 | Val NDCG@10 0.2950 | train 365s, eval 410s
Epoch  3 | loss 3.6944 | Val HR@10 0.4934 | Val NDCG@10 0.3226 | train 363s, eval 410s
Epoch  4 | loss 3.6149 | Val HR@10 0.5156 | Val NDCG@10 0.3391 | train 364s, eval 412s
Epoch  5 | loss 3.5578 | Val HR@10 0.5309 | Val NDCG@10 0.3507 | train 363s, eval 410s
Epoch  6 | loss 3.5162 | Val HR@10 0.5415 | Val NDCG@10 0.3591 | train 363s, eval 410s
Epoch  7 | loss 3.4810 | Val HR@10 0.5512 | Val NDCG@10 0.3653 | train 362s, eval 411s
Epoch  8 | loss 3.4529 | Val HR@10 0.5605 | Val NDCG@10 0.3722 | train 361s, eval 410s
Epoch  9 | loss 3.4287 | Val HR@10 0.5657 | Val NDCG@10 0.3775 | train 361s, eval 411s
Epoch 10 | loss 3.4077 | Val